In [1]:
import os 
import sys
os.environ['PYSPARK_PYTHON']='python'

In [2]:
from pyspark import SparkContext, SparkConf

In [3]:
conf = SparkConf().setAppName("RDD_LAB_2").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf = conf)
sc.setLogLevel("Error")

In [4]:
print("Spark Version:", sc.version)
print("App Name:", sc.appName)

Spark Version: 3.5.8
App Name: RDD_LAB_2


In [5]:
import warnings

In [6]:
warnings.filterwarnings('ignore')

In [7]:
numbers = [10,25,4,38,17,52,6,91,3,44]
rdd_nums = sc.parallelize(numbers)

print("Type:", type(rdd_nums))
print("Partition Count:", rdd_nums.getNumPartitions())
print("Count:", rdd_nums.count())
print("First 5:", rdd_nums.take(5))

Type: <class 'pyspark.rdd.RDD'>
Partition Count: 8
Count: 10
First 5: [10, 25, 4, 38, 17]


In [8]:
employees = [
    "Ram,Engineering,72000",
    "Hari,Marketing,55000",
    "Sita,Engineering,88000",
    "Gita,HR,49000",
    "Adam,Engineering,95000",
    "Eve,Marketing,61000",
    "Harry,HR,52000",
    "Hiro,Engineering,79000"
]

In [9]:
rdd_emp = sc.parallelize(employees)
print("Employee Count", rdd_emp.count())
print("First record", rdd_emp.first())

Employee Count 8
First record Ram,Engineering,72000


In [10]:
rdd_range = sc.range(1,21)

In [11]:
print(rdd_range)

PythonRDD[8] at RDD at PythonRDD.scala:53


In [12]:
rdd_range.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

In [13]:
print("Range Sum:", rdd_range.sum())

Range Sum: 210


In [14]:
import os
import shutil
import stat

def remove_readonly(func, path, excinfo):
    """Clear the read-only bit and retry the removal."""
    os.chmod(path, stat.S_IWRITE)
    func(path)

emp_path = "lab_output1/employees"

if os.path.exists(emp_path):
    # Changed 'onexc' to 'onerror' for Python 3.11 compatibility
    shutil.rmtree(emp_path, onerror=remove_readonly)

In [15]:
rdd_emp.saveAsTextFile(emp_path)
print("Employees Saved To:",emp_path)

Employees Saved To: lab_output1/employees


In [16]:
files = os.listdir(emp_path)
part_files = [f for f in files if f.startswith("part")]

print("Files written:", files)
print("Number of part files:", len(part_files))

Files written: ['.part-00000.crc', '.part-00001.crc', '.part-00002.crc', '.part-00003.crc', '.part-00004.crc', '.part-00005.crc', '.part-00006.crc', '.part-00007.crc', '._SUCCESS.crc', 'part-00000', 'part-00001', 'part-00002', 'part-00003', 'part-00004', 'part-00005', 'part-00006', 'part-00007', '_SUCCESS']
Number of part files: 8


In [17]:
# Use a wildcard (*) to grab only the files starting with 'part-'
rdd_emp_loaded = sc.textFile("lab_output1/employees/part-*")
rdd_emp_loaded.collect()

['Ram,Engineering,72000',
 'Hari,Marketing,55000',
 'Sita,Engineering,88000',
 'Gita,HR,49000',
 'Adam,Engineering,95000',
 'Eve,Marketing,61000',
 'Harry,HR,52000',
 'Hiro,Engineering,79000']

In [18]:
def parse_row(line):
    parts = line.split(",")
    name = parts[0]
    dept = parts[1]
    salary = int(parts[2])
    return (name,dept,salary)

In [19]:
rdd_parsed = rdd_emp_loaded.map(parse_row)
rdd_parsed.collect()

[('Ram', 'Engineering', 72000),
 ('Hari', 'Marketing', 55000),
 ('Sita', 'Engineering', 88000),
 ('Gita', 'HR', 49000),
 ('Adam', 'Engineering', 95000),
 ('Eve', 'Marketing', 61000),
 ('Harry', 'HR', 52000),
 ('Hiro', 'Engineering', 79000)]

In [20]:
rdd_eng = rdd_parsed.filter(lambda r: r[1] == "Engineering")
print("Engineering Count: ", rdd_eng.count())

Engineering Count:  4


In [21]:
rdd_hr = rdd_parsed.filter(lambda r: r[1] == "HR")
print("HR Count: ", rdd_hr.count())

HR Count:  2


In [22]:
rdd_eng_name = rdd_eng.map(lambda r: r[0])
print("Engineering Name: ", rdd_eng_name.collect())

Engineering Name:  ['Ram', 'Sita', 'Adam', 'Hiro']


In [23]:
rdd_salary = rdd_parsed.filter(lambda r: r[2] > 70000)
print("Salary more than 70,000", rdd_salary.collect())

Salary more than 70,000 [('Ram', 'Engineering', 72000), ('Sita', 'Engineering', 88000), ('Adam', 'Engineering', 95000), ('Hiro', 'Engineering', 79000)]


In [24]:
sc.stop()